# IAN 603 · Module 12
# Views & Modular Queries

This notebook is the hands-on, step-by-step companion for **Module 12: Views & Modular Queries**.

It follows the module's central idea:

> **Turn stable SQL logic into reusable interfaces, then choose whether the result should stay live or be stored as a snapshot.**

### Main topics
- Standard PostgreSQL views
- View contracts and grain
- `CREATE OR REPLACE VIEW`
- View definitions and dependencies
- Updatable views
- `WITH CHECK OPTION`
- Views as controlled analytics interfaces
- `security_invoker`
- Materialized views
- Refresh and staleness
- Indexing materialized views
- CTE vs view vs materialized view
- Modular analytics architecture

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module12.html



## Learning objectives

By the end of this notebook, you should be able to:

1. Explain a view as a reusable SQL **interface or contract**.
2. Create, query, and replace standard views.
3. Identify the intended **grain** of a view.
4. Inspect view definitions and dependencies before making changes.
5. Explain when a simple view may be updatable.
6. Use `WITH CHECK OPTION` to keep updates inside a view's filter.
7. Explain why views do not replace normal database permissions.
8. Distinguish a live standard view from a stored materialized view.
9. Demonstrate materialized-view staleness and refresh.
10. Choose among a CTE, standard view, and materialized view.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Build the Module 12 teaching database

Module 12 uses several small datasets because views are most useful when they provide stable interfaces across repeated joins, filters, calculations, and summaries.

| Base table | Grain |
|---|---|
| `patients` | one row per patient |
| `admissions` | one row per admission |
| `departments` | one row per department |
| `employees` | one row per employee |
| `sales_orders` | one row per sales order |
| `customer_transactions` | one row per transaction |

The setup also creates an `analytics` schema containing several views and one materialized view.


## 4. Create or reset all Module 12 objects

This setup is intentionally repeatable. Re-run it whenever you want a clean starting point.

It creates:
- base tables,
- sample rows,
- the `analytics` schema,
- reusable standard views,
- a materialized diagnosis summary.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 12\n-- Views & Modular Queries\n-- ============================================================\n-- Companion file for s_module12.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- HOW TO USE\n-- 1. Run SECTION 1 to recreate the sample schema and views.\n-- 2. Run later sections independently.\n-- 3. Some examples intentionally demonstrate behavior such as\n--    CHECK OPTION failures or stale materialized-view results.\n--\n-- CORE IDEAS\n-- * Standard view: stored query definition; data stays live.\n-- * View contract: column names, types, meaning, and row grain.\n-- * Updatable view: some simple views can pass writes to base table.\n-- * CHECK OPTION: prevents a write from making a row disappear\n--   from the view's own predicate.\n-- * Materialized view: stores query results and must be refreshed.\n-- * Dependency: downstream objects depend on upstream definitions.\n-- ============================================================\n\n-- ============================================================\n-- TABLE / OBJECT GUIDE\n-- ============================================================\n-- patients                 : one row per patient\n-- admissions               : one row per hospital admission\n-- departments              : one row per department\n-- employees                : one row per employee\n-- sales_orders             : one row per sales order\n-- customer_transactions    : one row per transaction\n-- analytics.vw_*           : reusable logical interfaces\n-- analytics.mv_*           : stored analytical snapshot\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATA\n-- ============================================================\n\nDROP SCHEMA IF EXISTS analytics CASCADE;\nDROP TABLE IF EXISTS customer_transactions CASCADE;\nDROP TABLE IF EXISTS sales_orders CASCADE;\nDROP TABLE IF EXISTS employees CASCADE;\nDROP TABLE IF EXISTS departments CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE SCHEMA analytics;\n\nCREATE TABLE patients (\n  subject_id integer PRIMARY KEY,\n  name text NOT NULL,\n  date_of_birth date NOT NULL,\n  gender text NOT NULL,\n  email text,\n  city text,\n  active boolean NOT NULL DEFAULT true\n);\n\nCREATE TABLE admissions (\n  hadm_id integer PRIMARY KEY,\n  subject_id integer NOT NULL REFERENCES patients(subject_id),\n  admit_time timestamp NOT NULL,\n  discharge_time timestamp,\n  department text NOT NULL,\n  diagnosis text NOT NULL\n);\n\nCREATE TABLE departments (\n  department_id integer PRIMARY KEY,\n  department_name text NOT NULL UNIQUE\n);\n\nCREATE TABLE employees (\n  employee_id integer PRIMARY KEY,\n  employee_name text NOT NULL,\n  department_id integer NOT NULL REFERENCES departments(department_id),\n  salary numeric(12,2) NOT NULL CHECK (salary > 0)\n);\n\nCREATE TABLE sales_orders (\n  order_id integer PRIMARY KEY,\n  employee_id integer NOT NULL REFERENCES employees(employee_id),\n  order_date date NOT NULL,\n  sale_amount numeric(12,2) NOT NULL CHECK (sale_amount >= 0)\n);\n\nCREATE TABLE customer_transactions (\n  transaction_id integer PRIMARY KEY,\n  customer_id integer NOT NULL,\n  transaction_date date NOT NULL,\n  amount numeric(12,2) NOT NULL CHECK (amount >= 0),\n  status text NOT NULL CHECK (status IN ('completed','pending','cancelled'))\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith','1980-04-16','Female','alice@example.org','Greensboro',true),\n(1002,'Bob Johnson','1993-02-02','Male','bob@example.org','Greensboro',true),\n(1003,'Carol White','1967-11-21','Female','carol@example.org','High Point',true),\n(1004,'David Brown','1958-06-30','Male','david@example.org','Winston-Salem',false),\n(1005,'Eve Davis','1996-09-08','Female','eve@example.org','Greensboro',true),\n(1006,'Grace Wilson','1973-01-19','Female','grace@example.org','Burlington',true);\n\nINSERT INTO admissions VALUES\n(101,1001,'2025-01-10 08:00','2025-01-15 12:00','Internal Medicine','Influenza'),\n(102,1002,'2025-02-01 14:00','2025-02-03 09:00','Orthopedics','Fracture'),\n(103,1001,'2025-03-05 10:00','2025-03-12 15:00','Pulmonology','Pneumonia'),\n(104,1003,'2025-04-10 07:00','2025-04-11 11:00','Cardiology','Hypertension'),\n(105,1001,'2025-05-15 09:00','2025-05-20 14:00','Internal Medicine','Influenza'),\n(106,1002,'2025-02-20 09:30','2025-02-22 10:00','Orthopedics','Follow-up'),\n(107,1006,'2025-05-01 07:30','2025-05-04 11:30','Pulmonology','Asthma'),\n(108,1006,'2025-05-25 10:00','2025-05-28 13:00','Pulmonology','Asthma');\n\nINSERT INTO departments VALUES\n(101,'Sales'),(102,'Marketing'),(103,'Analytics');\n\nINSERT INTO employees VALUES\n(1,'Jane Doe',101,92000),(2,'John Smith',101,78000),(3,'Emily Jones',102,81000),\n(4,'Ravi Patel',103,99000),(5,'Mina Park',103,94000);\n\nINSERT INTO sales_orders VALUES\n(1,1,'2025-01-10',5000),(2,1,'2025-02-12',3000),(3,2,'2025-02-20',4000),\n(4,3,'2025-03-05',2000),(5,4,'2025-03-20',6500),(6,4,'2025-04-11',3500);\n\nINSERT INTO customer_transactions VALUES\n(1,201,'2025-01-10',100,'completed'),(2,201,'2025-03-15',150,'completed'),\n(3,202,'2025-02-01',200,'completed'),(4,203,'2024-12-01',80,'completed'),\n(5,204,'2025-06-20',120,'completed'),(6,202,'2025-06-25',55,'pending');\n\nCREATE VIEW analytics.vw_patient_admission_summary AS\nSELECT\n  p.subject_id,\n  p.gender,\n  p.city,\n  a.hadm_id,\n  a.admit_time,\n  a.discharge_time,\n  a.department,\n  a.diagnosis,\n  CASE WHEN a.discharge_time IS NULL THEN NULL\n       ELSE ROUND(EXTRACT(EPOCH FROM (a.discharge_time - a.admit_time))/86400.0,2)\n  END AS stay_days\nFROM patients AS p\nJOIN admissions AS a ON a.subject_id = p.subject_id;\n\nCREATE VIEW analytics.vw_active_patients AS\nSELECT subject_id, name, date_of_birth, gender, email, city, active\nFROM patients\nWHERE active = true\nWITH LOCAL CHECK OPTION;\n\nCREATE VIEW analytics.vw_patient_analytics AS\nSELECT\n  subject_id,\n  CASE\n    WHEN date_of_birth > DATE '1995-06-30' THEN 'Under 30'\n    WHEN date_of_birth > DATE '1975-06-30' THEN '30-49'\n    WHEN date_of_birth > DATE '1955-06-30' THEN '50-69'\n    ELSE '70+'\n  END AS age_band,\n  city,\n  active\nFROM patients;\n\nCREATE VIEW analytics.vw_employee_performance AS\nSELECT\n  e.employee_id,\n  e.employee_name,\n  d.department_name,\n  COUNT(o.order_id) AS orders_processed,\n  COALESCE(SUM(o.sale_amount),0)::numeric(12,2) AS total_sales\nFROM employees AS e\nJOIN departments AS d ON d.department_id = e.department_id\nLEFT JOIN sales_orders AS o ON o.employee_id = e.employee_id\nGROUP BY e.employee_id, e.employee_name, d.department_name;\n\nCREATE VIEW analytics.vw_recent_customer_activity AS\nSELECT customer_id, transaction_date, amount\nFROM customer_transactions\nWHERE status = 'completed'\n  AND transaction_date >= DATE '2025-06-30' - INTERVAL '6 months';\n\nCREATE MATERIALIZED VIEW analytics.mv_diagnosis_stats AS\nSELECT\n  diagnosis,\n  COUNT(*) AS total_admissions,\n  COUNT(DISTINCT subject_id) AS unique_patients,\n  ROUND(AVG(CASE WHEN discharge_time IS NULL THEN NULL ELSE EXTRACT(EPOCH FROM (discharge_time - admit_time))/86400.0 END),2) AS avg_stay_days,\n  clock_timestamp() AS refreshed_at\nFROM admissions\nGROUP BY diagnosis;\n\nCREATE UNIQUE INDEX ux_mv_diagnosis_stats_diagnosis\nON analytics.mv_diagnosis_stats (diagnosis);\n\n"

execute_script(setup_sql)


## 5. Inspect the created database objects


In [ ]:
display_query("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema IN ('public', 'analytics')
ORDER BY table_schema, table_type, table_name;
""", title="Tables and views")


In [ ]:
display_query("""
SELECT schemaname, matviewname
FROM pg_matviews
WHERE schemaname = 'analytics'
ORDER BY matviewname;
""", title="Materialized views")


# Part II · Why Views Matter

A view is easiest to understand as a **named SQL interface**.

Suppose analysts repeatedly need patient and admission information.

The raw tables are separate:

**patients**

| subject_id | name | city |
|---:|---|---|
| 1001 | Alice | Greensboro |
| 1002 | Bob | Greensboro |

**admissions**

| hadm_id | subject_id | department |
|---:|---:|---|
| 101 | 1001 | Internal Medicine |
| 103 | 1001 | Pulmonology |
| 102 | 1002 | Orthopedics |

Without a view, every analyst must remember the same join.

With a view, we create a stable interface:

```text
patients + admissions
        ↓
analytics.vw_patient_admission_summary
        ↓
many downstream queries
```

The goal is not just shorter SQL. The goal is to make **shared logic consistent and reusable**.


## 6. Three reusable choices

Module 12 compares three ways to organize SQL logic.

| Tool | Lifetime | Stores rows? | Best use |
|---|---|---:|---|
| CTE | one statement | No | one query with several logical stages |
| Standard view | persistent database object | No | stable reusable live interface |
| Materialized view | persistent database object | Yes | reusable stored snapshot |

A good question to ask is:

> **How long should this logic live, and should its result be recalculated every time?**


# Part III · Standard Views

A standard view stores a **query definition**, not a separate copy of the result rows.

Conceptually:

```text
SELECT * FROM analytics.vw_patient_admission_summary
                  ↓
        PostgreSQL runs the view query
                  ↓
        current rows from base tables
```

This is why we call a standard view **live**.


## 7. Query an existing view

The view `analytics.vw_patient_admission_summary` combines patients and admissions and calculates length of stay.


In [ ]:
display_query("""
SELECT *
FROM analytics.vw_patient_admission_summary
ORDER BY subject_id, admit_time;
""", title="Patient-admission view")


### What is the grain?

Look at the columns:

| column | meaning |
|---|---|
| `hadm_id` | admission identifier |
| `subject_id` | patient identifier |
| `admit_time` | admission event |
| `stay_days` | measure for that admission |

Therefore the intended grain is:

> **one row per admission**

A view should have a clear row meaning, just like a table or CTE.


## 8. Validate the view's grain


In [ ]:
display_query("""
SELECT
    hadm_id,
    COUNT(*) AS rows_per_admission
FROM analytics.vw_patient_admission_summary
GROUP BY hadm_id
HAVING COUNT(*) > 1;
""", title="Check for duplicated admission IDs")


If this returns no rows, every admission appears at most once.

That does not prove the entire view is correct, but it is a useful **contract check**.


## 9. Build the view step by step

### Step 1 — Join the base tables

Before creating a view, verify the query that will define it.


In [ ]:
display_query("""
SELECT
    p.subject_id,
    p.gender,
    p.city,
    a.hadm_id,
    a.admit_time,
    a.discharge_time,
    a.department,
    a.diagnosis
FROM patients AS p
JOIN admissions AS a
    ON a.subject_id = p.subject_id
ORDER BY a.hadm_id;
""", title="Step 1: Verify the join")


### Step 2 — Add a reusable calculation

Instead of forcing every downstream query to recalculate stay length, put the calculation into the shared interface.

Small conceptual result:

| hadm_id | admit | discharge | stay_days |
|---:|---|---|---:|
| 101 | Jan 10 | Jan 15 | about 5.17 |
| 103 | Mar 05 | Mar 12 | about 7.21 |

### Step 3 — Turn the tested query into a view

```sql
CREATE OR REPLACE VIEW analytics.vw_patient_admission_summary AS
SELECT ...
```

After that, consumers can query the interface rather than repeat the join and calculation.


## 10. Create or replace the patient-admission view


In [ ]:
execute_query("""
CREATE OR REPLACE VIEW analytics.vw_patient_admission_summary AS
SELECT
    p.subject_id,
    p.gender,
    p.city,
    a.hadm_id,
    a.admit_time,
    a.discharge_time,
    a.department,
    a.diagnosis,
    CASE
        WHEN a.discharge_time IS NULL THEN NULL
        ELSE ROUND(
            EXTRACT(EPOCH FROM (a.discharge_time - a.admit_time)) / 86400.0,
            2
        )
    END AS stay_days
FROM patients AS p
JOIN admissions AS a
    ON a.subject_id = p.subject_id;
""")

print("View created or replaced.")


## 11. Reuse the view for a different analysis

The downstream query now starts from a clean reusable interface.


In [ ]:
display_query("""
SELECT
    department,
    COUNT(*) AS admissions,
    COUNT(DISTINCT subject_id) AS unique_patients,
    ROUND(AVG(stay_days), 2) AS avg_stay_days
FROM analytics.vw_patient_admission_summary
WHERE stay_days IS NOT NULL
GROUP BY department
ORDER BY admissions DESC, department;
""", title="Department summary from the shared view")


### Why this matters

Without the view, the analyst would need to know:
- how patients join to admissions,
- how stay length is calculated,
- which columns are considered part of the shared interface.

With the view, the analyst can focus on the new question.


# Part IV · View Contracts, Names, and Dependencies

A useful view is more than saved SQL. It is a **contract**.

A good contract communicates:
- what one row represents,
- which columns are exposed,
- which calculations are standardized,
- which downstream users can depend on it.

A name such as:

```text
analytics.vw_patient_admission_summary
```

also communicates:
- schema: `analytics`
- object type convention: `vw_`
- subject: patient admissions
- purpose: reusable summary/interface


## 12. Inspect view definitions


In [ ]:
display_query("""
SELECT
    table_schema,
    table_name,
    view_definition
FROM information_schema.views
WHERE table_schema = 'analytics'
ORDER BY table_name;
""", title="Standard view definitions")


## 13. Inspect dependencies

Before changing a shared view, inspect what base tables it depends on.


In [ ]:
display_query("""
SELECT
    view_schema,
    view_name,
    table_schema,
    table_name
FROM information_schema.view_table_usage
WHERE view_schema = 'analytics'
ORDER BY view_name, table_name;
""", title="View-table dependencies")


### Dependency mental model

```text
patients ─────┐
              ├── vw_patient_admission_summary ── downstream reports
admissions ───┘

employees ────┐
departments ──┼── vw_employee_performance ─────── analyst queries
sales_orders ─┘
```

Changing a base table or shared view can affect everything to the right.

That is why modular SQL also requires **change management**.


# Part V · Updatable Views

Some simple views can support `INSERT`, `UPDATE`, or `DELETE`.

But do not assume every view is writable.

Views containing complex joins, grouping, aggregates, or other transformations often cannot map an update cleanly back to one base row.

A simple filtered single-table view is easier for PostgreSQL to update.


## 14. The active-patient view

The view is based on one table and keeps only rows where:

```sql
active = true
```

Conceptually:

**patients**

| subject_id | name | active |
|---:|---|---|
| 1001 | Alice | true |
| 1004 | David | false |

**vw_active_patients**

| subject_id | name | active |
|---:|---|---|
| 1001 | Alice | true |

The view is a filtered window into the base table.


In [ ]:
display_query("""
SELECT
    subject_id,
    name,
    city,
    active
FROM analytics.vw_active_patients
ORDER BY subject_id;
""", title="Active-patient view")


## 15. Update through a simple view

A normal update through this view can modify the underlying `patients` row.

To keep the notebook repeatable, this example changes the city and immediately changes it back.


In [ ]:
display_query("""
UPDATE analytics.vw_active_patients
SET city = 'High Point'
WHERE subject_id = 1002
RETURNING subject_id, name, city, active;
""", title="Update through the view")


In [ ]:
execute_query("""
UPDATE patients
SET city = 'Greensboro'
WHERE subject_id = 1002;
""")

print("Reset subject_id 1002 to the original city.")


# Part VI · WITH CHECK OPTION

A filtered updatable view raises an important question:

> What happens if an update makes a row no longer satisfy the view's filter?

The Module 12 view uses:

```sql
WITH LOCAL CHECK OPTION
```

That means updates through the view must keep the resulting row valid for the view condition.

For `vw_active_patients`:

```text
before update: active = true  → visible in view
attempt:       active = false → would leave the view
CHECK OPTION:  reject the change
```


## 16. Intentional CHECK OPTION failure

The next cell is expected to produce a PostgreSQL error. That error is the lesson.

It demonstrates that the view's contract is enforced for updates made through the view.


In [ ]:
from sqlalchemy import text

try:
    with db.engine.begin() as conn:
        conn.execute(text("""
            UPDATE analytics.vw_active_patients
            SET active = false
            WHERE subject_id = 1002;
        """))

    print("Unexpected: the update succeeded.")

except Exception as e:
    print("✅ Expected PostgreSQL rejection.")
    print("WITH CHECK OPTION prevented the row from leaving the view.")
    print()
    print(str(e).split("\n")[0])



### Important

An error here does **not** mean the notebook is broken.

It means PostgreSQL prevented a row from being updated through the view into a state that the view itself would reject.


# Part VII · Views as Analytics Interfaces

A view can expose a deliberately limited set of columns.

For example, analysts may need:
- patient identifier,
- age band,
- city,
- active status,

but not necessarily the raw date of birth or email address.

This creates a cleaner interface for the analytical task.


## 17. Query the limited patient analytics view


In [ ]:
display_query("""
SELECT *
FROM analytics.vw_patient_analytics
ORDER BY subject_id;
""", title="Limited patient analytics interface")


### Interface transformation

Raw patient row:

| subject_id | date_of_birth | email | city | active |
|---:|---|---|---|---|
| 1001 | 1980-04-16 | ... | Greensboro | true |

Analytics interface:

| subject_id | age_band | city | active |
|---:|---|---|---|
| 1001 | 30-49 | Greensboro | true |

The view can standardize both **column selection** and **derived features**.


## 18. `security_invoker`

PostgreSQL supports a `security_invoker` view option.

The key idea is that permissions for underlying relations are checked using the invoking user's privileges.

The view still does not replace a complete access-control strategy. Permissions on schemas, tables, functions, and views still matter.


In [ ]:
execute_query("""
CREATE OR REPLACE VIEW analytics.vw_patient_analytics
WITH (security_invoker = true) AS
SELECT
    subject_id,
    CASE
        WHEN date_of_birth > DATE '1995-06-30' THEN 'Under 30'
        WHEN date_of_birth > DATE '1975-06-30' THEN '30-49'
        WHEN date_of_birth > DATE '1955-06-30' THEN '50-69'
        ELSE '70+'
    END AS age_band,
    city,
    active
FROM patients;
""")

print("security_invoker view created or replaced.")


# Part VIII · Materialized Views

A materialized view is different from a standard view because PostgreSQL **stores the query result**.

### Standard view

```text
query view
   ↓
run underlying SQL now
   ↓
current result
```

### Materialized view

```text
refresh
   ↓
run underlying SQL
   ↓
store rows
   ↓
later queries read stored snapshot
```

This can make repeated analytical reads faster, but it introduces a new responsibility:

> **freshness**


## 19. Query the diagnosis materialized view

Its grain is:

> **one row per diagnosis**


In [ ]:
display_query("""
SELECT *
FROM analytics.mv_diagnosis_stats
ORDER BY total_admissions DESC, diagnosis;
""", title="Diagnosis materialized view")


### Snapshot example

Imagine the live admissions table currently contains:

| diagnosis | live admissions |
|---|---:|
| Influenza | 3 |

but the last materialized refresh stored:

| diagnosis | stored admissions |
|---|---:|
| Influenza | 2 |

Both can be correct at the same time:

- the table is current,
- the materialized view is a snapshot from an earlier refresh.


## 20. Create a controlled stale-snapshot demonstration

First we add one new Influenza admission to the live base table.

The materialized view does **not** automatically change.


In [ ]:
execute_query("""
INSERT INTO admissions (
    hadm_id,
    subject_id,
    admit_time,
    discharge_time,
    department,
    diagnosis
)
VALUES (
    999,
    1005,
    '2025-06-20 09:00',
    '2025-06-22 10:00',
    'Internal Medicine',
    'Influenza'
)
ON CONFLICT (hadm_id) DO NOTHING;
""")

print("Live admission row inserted if it was not already present.")


In [ ]:
display_query("""
SELECT
    'live admissions' AS source,
    COUNT(*)::bigint AS influenza_rows
FROM admissions
WHERE diagnosis = 'Influenza'

UNION ALL

SELECT
    'materialized snapshot',
    COALESCE(MAX(total_admissions), 0)::bigint
FROM analytics.mv_diagnosis_stats
WHERE diagnosis = 'Influenza';
""", title="Live data vs stored snapshot")


If the counts differ, the materialized view is **stale by design**.

That is not automatically a problem.

The correct question is:

> How fresh does this analytical object need to be for its users?


## 21. Refresh the materialized view


In [ ]:
execute_query("""
REFRESH MATERIALIZED VIEW analytics.mv_diagnosis_stats;
""")

display_query("""
SELECT
    diagnosis,
    total_admissions,
    unique_patients,
    avg_stay_days,
    refreshed_at
FROM analytics.mv_diagnosis_stats
WHERE diagnosis = 'Influenza';
""", title="Snapshot after refresh")


### Freshness is part of the contract

A materialized view should have an explicit refresh policy.

Examples:
- after a nightly ETL load,
- every hour,
- before a dashboard is published,
- manually after a known batch update.

The refresh schedule should follow the analytical requirement, not habit.


## 22. Index a materialized view

Because a materialized view stores rows, it can have indexes.

The Module 12 example creates a unique index on diagnosis.


In [ ]:
execute_query("""
CREATE UNIQUE INDEX IF NOT EXISTS ux_mv_diagnosis_stats_diagnosis
ON analytics.mv_diagnosis_stats (diagnosis);
""")

display_query("""
SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname = 'analytics'
  AND tablename = 'mv_diagnosis_stats';
""", title="Materialized-view indexes")


### Concurrent refresh note

PostgreSQL can support:

```sql
REFRESH MATERIALIZED VIEW CONCURRENTLY ...
```

when its requirements are satisfied, including an appropriate unique index.

The important concept for this module is:

- normal refresh may block readers while replacing the snapshot,
- concurrent refresh is designed to reduce that disruption,
- concurrent refresh has additional requirements and tradeoffs.


# Part IX · CTE vs View vs Materialized View

Use the simplest tool that matches the lifetime and freshness requirement.

| Question | CTE | View | Materialized view |
|---|:---:|:---:|:---:|
| Needed only in one statement? | ✅ | | |
| Reused by many queries? | | ✅ | ✅ |
| Should always reflect current base data? | | ✅ | |
| Should store a reusable snapshot? | | | ✅ |
| Needs explicit refresh? | | | ✅ |
| Useful for a multi-stage one-off analysis? | ✅ | | |

### Decision shortcut

```text
One statement?
    → CTE

Reusable live logic?
    → standard view

Reusable stored result with acceptable staleness?
    → materialized view
```


# Part X · View + CTE: Modular Layers

Views and CTEs are not competitors.

A common architecture is:

```text
base tables
    ↓
stable reusable view
    ↓
CTE for today's analysis
    ↓
final result
```

The view handles stable shared logic.

The CTE handles temporary question-specific logic.


## 23. Readmission analysis from the shared view


In [ ]:
display_query("""
WITH sequenced AS (
    SELECT
        subject_id,
        hadm_id,
        admit_time,
        LAG(discharge_time) OVER (
            PARTITION BY subject_id
            ORDER BY admit_time, hadm_id
        ) AS previous_discharge
    FROM analytics.vw_patient_admission_summary
),
readmissions AS (
    SELECT
        *,
        admit_time - previous_discharge AS gap
    FROM sequenced
    WHERE previous_discharge IS NOT NULL
)
SELECT
    subject_id,
    hadm_id,
    gap,
    gap <= INTERVAL '30 days' AS within_30_days
FROM readmissions
ORDER BY subject_id, hadm_id;
""", title="View + CTE readmission analysis")


### Layer responsibilities

| Layer | Responsibility |
|---|---|
| view | stable patient-admission interface |
| first CTE | sequence admissions |
| second CTE | calculate repeat-visit gap |
| final query | classify the analytical result |

This is a strong modular pattern: **stable logic below, temporary logic above**.


# Part XI · Project Architecture Examples

Module 12 includes several reusable analytics interfaces.

```text
employees + departments + sales_orders
                  ↓
      vw_employee_performance
                  ↓
         top-performer query
```

and:

```text
customer_transactions
          ↓
vw_recent_customer_activity
          ↓
retention CTE
          ↓
at-risk customers
```


## 24. Employee performance view


In [ ]:
execute_query("""
CREATE OR REPLACE VIEW analytics.vw_employee_performance AS
SELECT
    e.employee_id,
    e.employee_name,
    d.department_name,
    COUNT(o.order_id) AS orders_processed,
    COALESCE(SUM(o.sale_amount), 0)::numeric(12,2) AS total_sales
FROM employees AS e
JOIN departments AS d
    ON d.department_id = e.department_id
LEFT JOIN sales_orders AS o
    ON o.employee_id = e.employee_id
GROUP BY
    e.employee_id,
    e.employee_name,
    d.department_name;
""")

display_query("""
SELECT *
FROM analytics.vw_employee_performance
ORDER BY total_sales DESC, employee_id;
""", title="Employee performance view")


## 25. Consume the employee view


In [ ]:
display_query("""
SELECT *
FROM analytics.vw_employee_performance
WHERE total_sales >= 5000
ORDER BY total_sales DESC;
""", title="Top performers")


Notice that the consumer does not need to rebuild:
- employee/department join,
- employee/order join,
- order count,
- sales total.

Those rules are encapsulated by the view.


## 26. Recent customer activity view


In [ ]:
display_query("""
SELECT *
FROM analytics.vw_recent_customer_activity
ORDER BY customer_id, transaction_date;
""", title="Recent completed customer activity")


## 27. Retention analysis using a CTE above the view


In [ ]:
display_query("""
WITH last_activity AS (
    SELECT
        customer_id,
        MAX(transaction_date) AS last_transaction
    FROM analytics.vw_recent_customer_activity
    GROUP BY customer_id
),
at_risk AS (
    SELECT
        customer_id,
        last_transaction,
        DATE '2025-06-30' - last_transaction AS days_inactive
    FROM last_activity
    WHERE last_transaction < DATE '2025-06-30' - 60
)
SELECT *
FROM at_risk
ORDER BY days_inactive DESC, customer_id;
""", title="Retention analysis")


# Part XII · Catalog and Governance

Shared analytics objects become easier to manage when you can discover and inspect them.

A useful catalog query should distinguish:
- standard views,
- materialized views.


## 28. Catalog the analytics objects


In [ ]:
display_query("""
SELECT
    schemaname AS object_schema,
    viewname AS object_name,
    'view' AS object_type
FROM pg_views
WHERE schemaname = 'analytics'

UNION ALL

SELECT
    schemaname,
    matviewname,
    'materialized view'
FROM pg_matviews
WHERE schemaname = 'analytics'

ORDER BY object_type, object_name;
""", title="Analytics object catalog")


## Architecture review questions

Before promoting SQL into a reusable object, ask:

1. **What is the row grain?**
2. **Who will reuse it?**
3. **Should it be live or stored?**
4. **How fresh must the data be?**
5. **What base objects does it depend on?**
6. **What happens if the interface changes?**
7. **Should updates through the view be allowed?**
8. **What permissions are required?**
9. **Is an index useful if the result is materialized?**
10. **Can temporary logic stay in a CTE instead?**


# Part XIII · Practice

Use the same reasoning process for every task:

1. Write the desired **grain**.
2. Decide whether the logic should live for one statement or be reusable.
3. Decide whether results must be live or may be a snapshot.
4. Build and test the underlying `SELECT` first.
5. Only then create the reusable object.


## Practice 1 · Employee analytics view

Create a view with one row per employee containing:
- employee ID,
- employee name,
- department name,
- order count,
- total sales.

<details>
<summary>Hint</summary>

Join:
- `employees`
- `departments`
- `sales_orders`

Use a `LEFT JOIN` to orders if employees with zero orders should remain visible.

</details>


In [ ]:
practice_1 = """
-- Write your SQL here


"""

# Uncomment when ready:
# display_query(practice_1, title="Practice 1")


## Practice 2 · Restricted patient interface

Create a view exposing only:
- `subject_id`
- an `age_band`
- `city`
- `active`

Do not expose email.

<details>
<summary>Hint</summary>

Use a `CASE` expression to convert `date_of_birth` into age groups.

</details>


In [ ]:
practice_2 = """
-- Write your SQL here


"""


## Practice 3 · Dependency inspection

Use `information_schema` to identify which base tables are used by each view in the `analytics` schema.

<details>
<summary>Hint</summary>

Look at:

```text
information_schema.view_table_usage
```

</details>


In [ ]:
practice_3 = """
-- Write your SQL here


"""


## Practice 4 · Standard vs materialized view

Explain which you would choose for each scenario:

1. A dashboard that must always show current patient status.
2. A large summary refreshed every morning.
3. Logic used only inside one complex report query.
4. A stable employee performance interface used by several analysts.

<details>
<summary>Suggested reasoning</summary>

Think about:
- reuse,
- freshness,
- storage,
- refresh responsibility.

</details>


## Practice 5 · Demonstrate staleness

1. Record the current value in a materialized view.
2. Add or change a base-table row.
3. Compare live and materialized results.
4. Refresh.
5. Compare again.

The goal is to demonstrate **why refresh policy matters**.


# Part XIV · Assessment Worked Example

### Task

Create:

```text
analytics.vw_employee_department_info
```

with:
- `employee_name`
- `salary`
- `department_name`

Then query **only the view** to calculate average salary for the Sales department.

### Reasoning

1. The reusable interface needs **one row per employee**.
2. Department name comes from `departments`.
3. Employee name and salary come from `employees`.
4. Once the view exists, the downstream average should use the view, not rebuild the join.


In [ ]:
execute_query("""
CREATE OR REPLACE VIEW analytics.vw_employee_department_info AS
SELECT
    e.employee_name,
    e.salary,
    d.department_name
FROM employees AS e
JOIN departments AS d
    ON d.department_id = e.department_id;
""")

display_query("""
SELECT
    department_name,
    ROUND(AVG(salary), 2) AS avg_salary
FROM analytics.vw_employee_department_info
WHERE department_name = 'Sales'
GROUP BY department_name;
""", title="Assessment sample solution")


# Part XV · Quick Concept Check

### 1. Does a standard view store its result rows?
<details><summary>Show answer</summary>No. It stores the query definition and returns live results from the underlying data.</details>

### 2. Which object stores a snapshot?
<details><summary>Show answer</summary>A materialized view.</details>

### 3. What new responsibility comes with a materialized view?
<details><summary>Show answer</summary>Freshness and refresh policy.</details>

### 4. When is a CTE often better than a view?
<details><summary>Show answer</summary>When the logic is needed only inside one statement.</details>

### 5. Why inspect dependencies before replacing a shared view?
<details><summary>Show answer</summary>Because downstream objects or reports may depend on its columns, grain, or behavior.</details>

### 6. What does CHECK OPTION help enforce?
<details><summary>Show answer</summary>Updates through a filtered view must continue satisfying that view's condition.</details>

### 7. Does a view automatically solve database security?
<details><summary>Show answer</summary>No. Permissions and broader access design still matter.</details>


# Module 12 Summary

### Standard view
Use it when stable SQL logic should become a reusable **live interface**.

### Materialized view
Use it when the result should be **stored** and the project can define an acceptable refresh policy.

### CTE
Use it when a named stage is useful inside **one statement**.

### Keep these habits

1. Define the **grain** before creating the object.
2. Test the underlying `SELECT` first.
3. Treat a shared view as a **contract**.
4. Inspect definitions and dependencies before changing shared objects.
5. Do not assume a view is writable.
6. Understand what `CHECK OPTION` protects.
7. Treat materialized-view staleness as a design decision.
8. Define refresh policy explicitly.
9. Keep stable logic in reusable layers and question-specific logic in CTEs.
10. Use the simplest object that solves the problem.

## Next
**Module 13**
